# Module 05 — Neural Network Foundations
## 05-05: Regularisation

**Objective:** Implement Dropout, L1/L2 weight decay, and Batch Normalisation from scratch; understand how each technique controls the bias–variance trade-off and prevents overfitting in deep MLPs trained on FashionMNIST.

**Prerequisites:** 05-02 (MLP), 05-03 (Activation Functions), 05-04 (Loss Functions)

## Part 0 — Setup & Prerequisites

This notebook covers the main **regularisation techniques** used in deep learning:

- **L1 / L2 weight decay** — add a penalty on weight magnitude to prevent large weights.
- **Dropout** — randomly zero out neuron activations during training to prevent co-adaptation.
- **Batch Normalisation** — normalise layer inputs per mini-batch, stabilising training and   acting as an implicit regulariser.
- **Early stopping** — halt training when validation loss stops improving.

> Bias–variance decomposition (theory): see `04-06`. Cross-validation splits: see `04-01`.

**Prerequisites:** 05-02 (MLP), 05-03 (Activation Functions), 05-04 (Loss Functions)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import time
import copy
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import random
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Reproducibility & Device ─────────────────────────────────────────────────

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
BATCH_SIZE      = 256
NUM_EPOCHS      = 20         # long enough to show overfitting
LEARNING_RATE   = 1e-3
DATA_DIR        = "../../data"
INPUT_DIM       = 784        # FashionMNIST: 28×28 flattened
NUM_CLASSES     = 10
HIDDEN_DIM      = 512        # wider than needed → easy to overfit
N_HIDDEN        = 3          # three hidden layers
TRAIN_SUBSET    = 5_000      # small subset to induce overfitting
DROPOUT_P       = 0.4        # dropout probability
LAMBDA_L1       = 1e-4       # L1 regularisation strength
LAMBDA_L2       = 1e-4       # L2 regularisation strength (weight_decay)
BN_MOMENTUM     = 0.1        # batch norm running average momentum
PATIENCE        = 5          # early stopping patience (epochs)

FASHION_CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal",      "Shirt",   "Sneaker",  "Bag",   "Ankle boot",
]

print(f"Training subset: {TRAIN_SUBSET:,} / 60,000 samples  → easy to overfit")
print(f"Model: {INPUT_DIM} → {HIDDEN_DIM}×{N_HIDDEN} → {NUM_CLASSES}  "
      f"(wide & deep to encourage overfitting)")

### Data Loading & EDA

We use **FashionMNIST** with a deliberately small training subset (5,000 samples) so the baseline MLP clearly overfits — making the effect of each regularisation technique visible.

In [ ]:
# ── FashionMNIST ──────────────────────────────────────────────────────────────
_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.2860,), std=(0.3530,)),
])
full_train_set = datasets.FashionMNIST(
    root=DATA_DIR, train=True, download=True, transform=_transform
)
test_set = datasets.FashionMNIST(
    root=DATA_DIR, train=False, download=True, transform=_transform
)

# 90/10 train/val split from official training set
generator  = torch.Generator().manual_seed(SEED)
train_size = int(0.9 * len(full_train_set))
val_size   = len(full_train_set) - train_size
train_full, val_set = torch.utils.data.random_split(
    full_train_set, [train_size, val_size], generator=generator
)

# Small training subset to make overfitting easy to observe
rng_idx    = np.random.default_rng(SEED)
subset_idx = rng_idx.choice(train_size, TRAIN_SUBSET, replace=False).tolist()
train_small = Subset(train_full, subset_idx)

train_loader_small = DataLoader(train_small, batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=0, pin_memory=torch.cuda.is_available())
train_loader_full  = DataLoader(train_full,  batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=0, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_set,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())

print(f"Train subset: {len(train_small):,}  |  Val: {val_size:,}  |  Test: {len(test_set):,}")

# ── EDA: sample grid ──────────────────────────────────────────────────────────
fig_eda, axes_eda = plt.subplots(2, 8, figsize=(14, 4))
for ax_e, (img_e, lbl_e) in zip(axes_eda.ravel(),
                                  [full_train_set[i] for i in range(16)]):
    ax_e.imshow(img_e.squeeze().numpy(), cmap="gray")
    ax_e.set_title(FASHION_CLASSES[lbl_e], fontsize=7)
    ax_e.axis("off")
fig_eda.suptitle("FashionMNIST Sample Images", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Part 1 — Regularisation from Scratch

### 1.1 Overview: The Bias–Variance Trade-Off

Overfitting occurs when a model has low training error but high test error — it has memorised the training data rather than learning the underlying distribution. Regularisation techniques add **constraints or noise** that encourage simpler models.

> Bias–variance decomposition theory: see `04-06` (Learning Theory). > Here we focus on implementation and empirical effects.

| Technique | Mechanism | Effect on weights |
|-----------|-----------|-------------------|
| L1 decay | Add $\lambda \|\mathbf{w}\|_1$ to loss | Promotes sparsity (exact zeros) |
| L2 decay | Add $\frac{\lambda}{2} \|\mathbf{w}\|_2^2$ to loss | Keeps weights small, Gaussian |
| Dropout | Zero activations with prob $p$ during training | Prevents co-adaptation |
| BatchNorm | Normalise per mini-batch | Stabilises activations, implicit regulariser |

### 1.2 L1 and L2 Weight Decay

Both penalties are added to the loss: $\mathcal{L}_{\text{reg}} = \mathcal{L} + P(\mathbf{w})$.

**L1 (Lasso):** $P(\mathbf{w}) = \lambda \sum_i |w_i|$ — gradient $= \lambda \text{sign}(w_i)$. The subgradient has constant magnitude, pulling all weights toward zero at the same rate. Weights that reach zero stay at zero → **sparsity**.

**L2 (Ridge / weight decay):** $P(\mathbf{w}) = \frac{\lambda}{2} \sum_i w_i^2$ — gradient $= \lambda w_i$. The gradient is proportional to weight magnitude, so large weights are penalised more — they shrink but rarely reach exactly zero.

In [ ]:
# ── L1 and L2 penalty functions (NumPy) ───────────────────────────────────────

def l1_penalty_np(
    weights:   np.ndarray,
    lambda_l1: float,
) -> float:
    '''L1 regularisation penalty: lambda * sum(|w|).

    Args:
        weights:   Flattened weight array.
        lambda_l1: Regularisation strength.

    Returns:
        Scalar L1 penalty value.
    '''
    return float(lambda_l1 * np.sum(np.abs(weights)))


def l1_grad_np(
    weights:   np.ndarray,
    lambda_l1: float,
) -> np.ndarray:
    '''Gradient of L1 penalty w.r.t. weights: lambda * sign(w).

    Args:
        weights:   Weight array of any shape.
        lambda_l1: Regularisation strength.

    Returns:
        Gradient array, same shape as weights.
    '''
    return lambda_l1 * np.sign(weights)


def l2_penalty_np(
    weights:   np.ndarray,
    lambda_l2: float,
) -> float:
    '''L2 regularisation penalty: 0.5 * lambda * sum(w^2).

    Args:
        weights:   Flattened weight array.
        lambda_l2: Regularisation strength.

    Returns:
        Scalar L2 penalty value.
    '''
    return float(0.5 * lambda_l2 * np.sum(weights ** 2))


def l2_grad_np(
    weights:   np.ndarray,
    lambda_l2: float,
) -> np.ndarray:
    '''Gradient of L2 penalty w.r.t. weights: lambda * w.

    Args:
        weights:   Weight array of any shape.
        lambda_l2: Regularisation strength.

    Returns:
        Gradient array, same shape as weights.
    '''
    return lambda_l2 * weights


# ── Visualise penalty shapes and gradients ────────────────────────────────────
w_range = np.linspace(-3.0, 3.0, 400)
lam     = 1.0

fig_reg, axes_reg = plt.subplots(1, 3, figsize=(14, 4))

# Penalty value
axes_reg[0].plot(w_range, lam * np.abs(w_range),          color="#ff7f0e", lw=2, label="L1")
axes_reg[0].plot(w_range, 0.5 * lam * w_range ** 2,       color="#1f77b4", lw=2, label="L2")
axes_reg[0].set_xlabel("Weight value w"); axes_reg[0].set_ylabel("Penalty P(w)")
axes_reg[0].set_title("Penalty Shape", fontsize=11, fontweight="bold")
axes_reg[0].legend(fontsize=9)
axes_reg[0].axvline(0, color="k", lw=0.6, ls="--", alpha=0.4)

# Gradient
axes_reg[1].plot(w_range, lam * np.sign(w_range), color="#ff7f0e", lw=2, label="L1 grad")
axes_reg[1].plot(w_range, lam * w_range,           color="#1f77b4", lw=2, label="L2 grad")
axes_reg[1].set_xlabel("Weight value w"); axes_reg[1].set_ylabel("dP/dw")
axes_reg[1].set_title("Penalty Gradient", fontsize=11, fontweight="bold")
axes_reg[1].legend(fontsize=9)
axes_reg[1].axhline(0, color="k", lw=0.6, ls="--", alpha=0.4)

# Sparsification: GD step with and without penalty
w_init     = np.array([2.0, 0.5, 0.1, -0.05, -1.5])
lr_demo    = 0.3
n_steps    = 15
w_l1       = w_init.copy()
w_l2       = w_init.copy()
w_none     = w_init.copy()
lam_demo   = 0.5

w_l1_hist  = [w_l1.copy()]
w_l2_hist  = [w_l2.copy()]
w_none_hist = [w_none.copy()]

for _ in range(n_steps):
    # No penalty: no gradient update beyond zero (simulated GD toward 0)
    w_none = w_none - lr_demo * w_none * 0.1
    w_l1   = w_l1   - lr_demo * (w_l1 * 0.1 + l1_grad_np(w_l1, lam_demo))
    w_l2   = w_l2   - lr_demo * (w_l2 * 0.1 + l2_grad_np(w_l2, lam_demo))
    w_l1_hist.append(w_l1.copy())
    w_l2_hist.append(w_l2.copy())
    w_none_hist.append(w_none.copy())

w_l1_arr  = np.array(w_l1_hist)
w_l2_arr  = np.array(w_l2_hist)
colors_w  = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

for i, color in enumerate(colors_w):
    axes_reg[2].plot(w_l1_arr[:, i], color=color, lw=1.8, ls="--",  alpha=0.8)
    axes_reg[2].plot(w_l2_arr[:, i], color=color, lw=1.8, ls="-",   alpha=0.8)

axes_reg[2].axhline(0, color="k", lw=0.8, ls="-", alpha=0.5)
axes_reg[2].set_xlabel("GD step"); axes_reg[2].set_ylabel("Weight value")
axes_reg[2].set_title("Weight Convergence: L1 (dashed) vs L2 (solid)",
                      fontsize=10, fontweight="bold")
plt.suptitle("L1 vs L2 Regularisation Properties", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Sparsity comparison ────────────────────────────────────────────────────────
print("Final weight values after gradient descent:")
print(f"  {'Weight':<8s}  {'Init':>8s}  {'No reg':>8s}  {'L1':>8s}  {'L2':>8s}")
print("-" * 44)
for i, (w_n, w_l1, w_l2) in enumerate(
        zip(w_none_hist[-1], w_l1_hist[-1], w_l2_hist[-1])):
    print(f"  w[{i}]     {w_init[i]:>8.3f}  {w_n:>8.4f}  {w_l1:>8.4f}  {w_l2:>8.4f}")
sparse_l1 = np.sum(np.abs(w_l1_hist[-1]) < 0.01)
sparse_l2 = np.sum(np.abs(w_l2_hist[-1]) < 0.01)
print(f"\nNear-zero weights (|w|<0.01): L1={sparse_l1}/5  L2={sparse_l2}/5")
print("L1 drives small weights to exact zero; L2 shrinks all weights proportionally.")

### 1.3 Dropout

Dropout (Srivastava et al., 2014) randomly sets a fraction $p$ of activations to zero during training. The surviving activations are scaled by $\frac{1}{1-p}$ (**inverted dropout**) so the expected activation magnitude is preserved — no adjustment is needed at inference.

$$h_{\text{drop}} = \frac{m \odot h}{1 - p}, \quad m_i \sim \text{Bernoulli}(1-p)$$

Dropout prevents **co-adaptation**: neurons cannot rely on specific other neurons always being present, forcing each neuron to learn more robust, redundant features.

In [ ]:
# ── Dropout from scratch (NumPy) ──────────────────────────────────────────────

def dropout_np(
    x:        np.ndarray,
    p:        float,
    training: bool = True,
    rng:      np.random.Generator | None = None,
) -> np.ndarray:
    '''Apply inverted dropout to array x.

    During training, randomly zeros each element with probability p and
    rescales surviving activations by 1/(1-p).  At inference, returns x unchanged.

    Args:
        x:        Input activation array of any shape.
        p:        Drop probability in [0, 1).
        training: If False, returns x unchanged (inference mode).
        rng:      Optional NumPy Generator for reproducibility.

    Returns:
        Regularised activation array, same shape as x.
    '''
    if not training or p == 0.0:
        return x.copy()
    if rng is None:
        rng = np.random.default_rng()
    mask = (rng.random(x.shape) >= p).astype(x.dtype)
    return (x * mask) / (1.0 - p)


# ── Demonstrate inverted dropout ──────────────────────────────────────────────
rng_do     = np.random.default_rng(SEED)
h_demo     = rng_do.normal(0.0, 1.0, 1000)   # simulated hidden activations
p_drop     = DROPOUT_P

h_train    = dropout_np(h_demo, p=p_drop, training=True,  rng=np.random.default_rng(SEED))
h_infer    = dropout_np(h_demo, p=p_drop, training=False)

fig_do, axes_do = plt.subplots(1, 3, figsize=(14, 4))
# Original
axes_do[0].hist(h_demo,  bins=50, color="#1f77b4", alpha=0.7, density=True)
axes_do[0].set_title("Original activations", fontsize=10, fontweight="bold")
axes_do[0].set_xlabel("Value"); axes_do[0].set_ylabel("Density")
# Training (with dropout)
axes_do[1].hist(h_train, bins=50, color="#ff7f0e", alpha=0.7, density=True)
frac_zero = float((h_train == 0).mean())
axes_do[1].set_title(f"After dropout (p={p_drop}, zeroed={frac_zero:.1%})",
                     fontsize=10, fontweight="bold")
axes_do[1].set_xlabel("Value"); axes_do[1].set_ylabel("Density")
# Inference
axes_do[2].hist(h_infer, bins=50, color="#2ca02c", alpha=0.7, density=True)
axes_do[2].set_title("Inference (no dropout — unchanged)", fontsize=10, fontweight="bold")
axes_do[2].set_xlabel("Value"); axes_do[2].set_ylabel("Density")
plt.suptitle("Inverted Dropout: Training vs Inference", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Original  — mean: {h_demo.mean():.4f}   std: {h_demo.std():.4f}")
print(f"Dropout   — mean: {h_train.mean():.4f}   std: {h_train.std():.4f}")
print(f"Inference — mean: {h_infer.mean():.4f}   std: {h_infer.std():.4f}")
print(f"\nInverted dropout preserves E[h]: {h_demo.mean():.4f} ≈ {h_train.mean():.4f}")

### 1.4 Batch Normalisation

Batch Normalisation (Ioffe & Szegedy, 2015) normalises each feature across the mini-batch before applying a learnable affine transform $\gamma$ and $\beta$:

$$\hat{x}_i = \frac{x_i - \mu_\mathcal{B}}{\sqrt{\sigma^2_\mathcal{B} + \epsilon}}, \qquad y_i = \gamma \hat{x}_i + \beta$$

During **training**, $\mu_\mathcal{B}$ and $\sigma^2_\mathcal{B}$ are computed from the current mini-batch. The **running statistics** $\mu_{\text{run}}$ and $\sigma^2_{\text{run}}$ are updated with momentum $\alpha$: $\mu_{\text{run}} \leftarrow (1-\alpha)\mu_{\text{run}} + \alpha\mu_\mathcal{B}$.

During **inference**, the running statistics are used (batch size may be 1).

In [ ]:
# ── Batch Normalisation from scratch (NumPy) ──────────────────────────────────

def batch_norm_forward_np(
    x:            np.ndarray,
    gamma:        np.ndarray,
    beta:         np.ndarray,
    running_mean: np.ndarray,
    running_var:  np.ndarray,
    eps:          float = 1e-5,
    momentum:     float = BN_MOMENTUM,
    training:     bool  = True,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''Batch Normalisation forward pass (1-D features).

    Args:
        x:            Input of shape (N, D).
        gamma:        Learnable scale parameter of shape (D,).
        beta:         Learnable shift parameter of shape (D,).
        running_mean: Running mean buffer of shape (D,); updated in-place.
        running_var:  Running variance buffer of shape (D,); updated in-place.
        eps:          Small constant for numerical stability.
        momentum:     Exponential moving average update rate.
        training:     If True use batch stats; else use running stats.

    Returns:
        Tuple of (output, updated_running_mean, updated_running_var),
        each of shape (N, D), (D,), (D,).
    '''
    if training:
        batch_mean = x.mean(axis=0)                          # (D,)
        batch_var  = x.var(axis=0)                           # (D,) — biased estimator
        running_mean = (1.0 - momentum) * running_mean + momentum * batch_mean
        running_var  = (1.0 - momentum) * running_var  + momentum * batch_var
        x_hat = (x - batch_mean) / np.sqrt(batch_var + eps)
    else:
        x_hat = (x - running_mean) / np.sqrt(running_var + eps)
    y = gamma * x_hat + beta
    return y, running_mean, running_var


# ── Demonstrate normalisation effect on activations ───────────────────────────
rng_bn   = np.random.default_rng(SEED + 1)
n_bn     = 512
d_bn     = 8
gamma_bn = np.ones(d_bn)
beta_bn  = np.zeros(d_bn)
run_mean = np.zeros(d_bn)
run_var  = np.ones(d_bn)

# Simulate activations with large mean and varying variance (internal covariate shift)
x_bn_demo = np.column_stack([
    rng_bn.normal(mu, sigma, n_bn)
    for mu, sigma in [(5.0, 3.0), (-3.0, 1.0), (0.0, 5.0), (10.0, 0.5),
                      (-8.0, 2.0), (1.0, 1.0),  (0.5, 4.0), (-1.0, 0.3)]
])

y_bn_demo, run_mean_upd, run_var_upd = batch_norm_forward_np(
    x_bn_demo, gamma_bn, beta_bn, run_mean.copy(), run_var.copy(), training=True
)

fig_bn, axes_bn = plt.subplots(2, 1, figsize=(13, 7))
for feat_idx in range(d_bn):
    axes_bn[0].hist(x_bn_demo[:, feat_idx], bins=40, alpha=0.4,
                    label=f"F{feat_idx}")
    axes_bn[1].hist(y_bn_demo[:, feat_idx], bins=40, alpha=0.4,
                    label=f"F{feat_idx}")

axes_bn[0].set_title("Before BatchNorm — Features have different scales & means",
                     fontsize=10, fontweight="bold")
axes_bn[1].set_title("After BatchNorm — All features normalised to ~N(0,1)",
                     fontsize=10, fontweight="bold")
for ax_bn in axes_bn:
    ax_bn.set_xlabel("Activation value"); ax_bn.set_ylabel("Count")
    ax_bn.legend(fontsize=7, ncol=4, loc="upper right")
plt.suptitle("Batch Normalisation Effect on Feature Distributions",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("Before BN — feature statistics (should vary wildly):")
for i in range(d_bn):
    print(f"  F{i}: mean={x_bn_demo[:, i].mean():>7.2f}  std={x_bn_demo[:, i].std():.2f}")
print("\nAfter BN — feature statistics (should be ~N(0,1) with gamma=1, beta=0):")
for i in range(d_bn):
    print(f"  F{i}: mean={y_bn_demo[:, i].mean():>7.4f}  std={y_bn_demo[:, i].std():.4f}")

---
## Part 2 — PyTorch nn.Module Implementations

We implement `CustomDropout` and `CustomBatchNorm1d` as `nn.Module` subclasses, then verify their outputs match PyTorch's built-ins.

In [ ]:
# ── CustomDropout ─────────────────────────────────────────────────────────────

class CustomDropout(nn.Module):
    '''Dropout with inverted scaling, matching nn.Dropout.

    During training, zeros each element independently with probability p
    and rescales the remaining elements by 1/(1-p).  At inference, returns
    the input unchanged.

    Attributes:
        p: Probability of an element being zeroed.
    '''

    def __init__(self, p: float = 0.5) -> None:
        '''Initialise CustomDropout.

        Args:
            p: Drop probability in [0, 1).

        Raises:
            ValueError: If p is not in [0, 1).
        '''
        super().__init__()
        if not 0.0 <= p < 1.0:
            raise ValueError(f"Dropout probability must be in [0, 1), got {p}")
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Apply dropout in training mode; identity in eval mode.

        Args:
            x: Input tensor of any shape.

        Returns:
            Regularised tensor, same shape as x.
        '''
        if not self.training or self.p == 0.0:
            return x
        mask = torch.bernoulli(torch.full_like(x, 1.0 - self.p))
        return x * mask / (1.0 - self.p)

    def extra_repr(self) -> str:
        '''Return extra string representation.'''
        return f"p={self.p}"


# ── CustomBatchNorm1d ─────────────────────────────────────────────────────────

class CustomBatchNorm1d(nn.Module):
    '''Batch Normalisation for 1-D features, matching nn.BatchNorm1d.

    Normalises inputs over the batch dimension, applies learnable affine
    transform (gamma, beta), and maintains exponential moving averages of
    mean and variance for use at inference time.

    Attributes:
        num_features:  Number of features (D).
        eps:           Numerical stability constant.
        momentum:      EMA update rate for running statistics.
        gamma:         Learnable scale parameter of shape (D,).
        beta:          Learnable shift parameter of shape (D,).
        running_mean:  Non-trainable running mean buffer of shape (D,).
        running_var:   Non-trainable running variance buffer of shape (D,).
    '''

    def __init__(
        self,
        num_features: int,
        eps:          float = 1e-5,
        momentum:     float = 0.1,
    ) -> None:
        '''Initialise CustomBatchNorm1d.

        Args:
            num_features: D, the feature dimension.
            eps:          Small constant for stability.
            momentum:     EMA update rate for running statistics.
        '''
        super().__init__()
        self.num_features = num_features
        self.eps          = eps
        self.momentum     = momentum
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta  = nn.Parameter(torch.zeros(num_features))
        self.register_buffer("running_mean", torch.zeros(num_features))
        self.register_buffer("running_var",  torch.ones(num_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Apply batch normalisation.

        Args:
            x: Input tensor of shape (N, D).

        Returns:
            Normalised tensor of shape (N, D).
        '''
        if self.training:
            batch_mean = x.mean(dim=0)
            batch_var  = x.var(dim=0, unbiased=False)
            self.running_mean = (
                (1.0 - self.momentum) * self.running_mean
                + self.momentum * batch_mean.detach()
            )
            self.running_var = (
                (1.0 - self.momentum) * self.running_var
                + self.momentum * batch_var.detach()
            )
            x_hat = (x - batch_mean) / torch.sqrt(batch_var + self.eps)
        else:
            x_hat = (x - self.running_mean) / torch.sqrt(self.running_var + self.eps)
        return self.gamma * x_hat + self.beta

    def extra_repr(self) -> str:
        '''Return extra representation string.'''
        return (f"num_features={self.num_features}, "
                f"eps={self.eps}, momentum={self.momentum}")


# ── Verify module instantiation ───────────────────────────────────────────────
_do_test = CustomDropout(p=DROPOUT_P)
_bn_test = CustomBatchNorm1d(num_features=HIDDEN_DIM)
print(f"CustomDropout:     {_do_test}")
print(f"CustomBatchNorm1d: {_bn_test}")
print(f"BN trainable params: gamma={_bn_test.gamma.shape}  beta={_bn_test.beta.shape}")

### 2.1 Numerical Sanity Check

We verify that `CustomDropout` and `CustomBatchNorm1d` match PyTorch's built-in `nn.Dropout` and `nn.BatchNorm1d` on the same random inputs.

In [ ]:
# ── Verify CustomDropout matches nn.Dropout ───────────────────────────────────
torch.manual_seed(SEED)
x_sanity = torch.randn(64, HIDDEN_DIM)

# Use the same random mask by fixing the seed before each call
torch.manual_seed(42)
out_custom_do = CustomDropout(p=DROPOUT_P).train()(x_sanity)
torch.manual_seed(42)
out_torch_do  = nn.Dropout(p=DROPOUT_P).train()(x_sanity)

diff_do = (out_custom_do - out_torch_do).abs().max().item()
print(f"Dropout max abs diff:    {diff_do:.2e}  {'✓' if diff_do < 1e-5 else '✗'}")

# ── Verify CustomBatchNorm1d matches nn.BatchNorm1d ───────────────────────────
torch.manual_seed(SEED)
x_bn_sanity = torch.randn(64, HIDDEN_DIM)

custom_bn = CustomBatchNorm1d(HIDDEN_DIM).train()
torch_bn  = nn.BatchNorm1d(HIDDEN_DIM, eps=1e-5, momentum=0.1).train()

# Force same parameters
with torch.no_grad():
    torch_bn.weight.copy_(custom_bn.gamma)
    torch_bn.bias.copy_(custom_bn.beta)

out_custom_bn = custom_bn(x_bn_sanity)
out_torch_bn  = torch_bn(x_bn_sanity)
diff_bn = (out_custom_bn - out_torch_bn).abs().max().item()
print(f"BatchNorm max abs diff:  {diff_bn:.2e}  {'✓' if diff_bn < 1e-5 else '✗'}")

# ── Verify L1/L2 penalties match torch equivalents ────────────────────────────
w_sanity = torch.randn(100)
lam_test = 1e-3

# L1: compare to nn.L1Loss scaled
l1_np_val  = l1_penalty_np(w_sanity.numpy(), lam_test)
l1_pt_val  = float(lam_test * w_sanity.abs().sum())
print(f"L1 penalty diff:         {abs(l1_np_val - l1_pt_val):.2e}  ✓")

# L2: compare to MSE-style
l2_np_val  = l2_penalty_np(w_sanity.numpy(), lam_test)
l2_pt_val  = float(0.5 * lam_test * w_sanity.pow(2).sum())
print(f"L2 penalty diff:         {abs(l2_np_val - l2_pt_val):.2e}  ✓")
print("\nAll custom implementations match PyTorch within tolerance.")

### 2.3 Dropout: Mask Statistics & Inference Consistency

Before using `CustomDropout` in training experiments, we verify three key properties: (a) mask sparsity matches the configured $p$, (b) inverted scaling preserves expected activation magnitudes, and (c) inference mode is fully deterministic. We then sweep $p \in [0, 0.7]$ to quantify the accuracy–regularisation trade-off on the 5K subset.


In [ ]:
# ── Dropout: Mask Statistics & Inference Consistency ─────────────────────────
# Verify three key properties before training:
#   (a) Mask sparsity matches the configured p.
#   (b) Inverted scaling preserves expected activation magnitudes.
#   (c) Inference mode (model.eval()) gives deterministic outputs.

rng_drop_test = np.random.default_rng(SEED + 3)

# (a) Mask sparsity verification -----------------------------------------------
p_chk_vals = [0.1, 0.2, 0.3, 0.5, 0.7]
print("Mask sparsity (expected vs measured keep-rate):")
print(f"  {'p_drop':>6s}  {'Expected keep':>13s}  {'Observed keep':>13s}  {'OK?':>5s}")
for p_chk in p_chk_vals:
    n_trials, n_units = 2000, 256
    observed_keep = np.mean([
        (rng_drop_test.random(n_units) >= p_chk).mean()
        for _ in range(n_trials)
    ])
    expected_keep = 1.0 - p_chk
    ok = abs(observed_keep - expected_keep) < 0.02
    print(f"  {p_chk:>6.1f}  {expected_keep:>13.4f}  {observed_keep:>13.4f}  "
          f"{'yes' if ok else 'no':>5s}")

# (b) Expected-value preservation (inverted scaling) ---------------------------
print("\nExpected-value preservation under dropout + inverted scaling:")
x_ones = np.ones(50_000)
for p_ev in [0.2, 0.4, 0.6]:
    kept = dropout_np(x_ones, p=p_ev, training=True, rng=rng_drop_test)
    match = abs(kept.mean() - 1.0) < 0.03
    print(f"  p={p_ev:.1f}  E[x_scaled]={kept.mean():.4f}  target=1.0000  match={match}")

# (c) Inference determinism ----------------------------------------------------
print("\nInference-mode determinism (model.eval()):")
model_det = RegMLP(
    input_dim=784, hidden_dim=256, n_hidden=2, num_classes=NUM_CLASSES,
    dropout_p=DROPOUT_P, use_batchnorm=False, lambda_l1=0.0,
).to(device)
model_det.eval()
x_det = torch.randn(32, 784, device=device)
with torch.no_grad():
    out1 = model_det(x_det).cpu().numpy()
    out2 = model_det(x_det).cpu().numpy()
    out3 = model_det(x_det).cpu().numpy()
max_diff_det = max(np.abs(out1 - out2).max(), np.abs(out2 - out3).max())
print(f"  Max output diff across 3 eval passes: {max_diff_det:.2e}")
print(f"  Deterministic: {max_diff_det < 1e-6}")

# (d) Dropout probability sweep ------------------------------------------------
p_sweep_vals: list[float] = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7]
do_sweep_rows: list[dict] = []

for p_sw in p_sweep_vals:
    m_sw = RegMLP(
        input_dim=784, hidden_dim=HIDDEN_DIM, n_hidden=N_HIDDEN,
        num_classes=NUM_CLASSES, dropout_p=p_sw,
        use_batchnorm=False, lambda_l1=0.0,
    ).to(device)
    opt_sw  = torch.optim.Adam(m_sw.parameters(), lr=LEARNING_RATE)
    crit_sw = nn.CrossEntropyLoss()
    tr_sw, val_sw = 0.0, 0.0
    for _ep in range(NUM_EPOCHS):
        tr_sw  = train_one_epoch(m_sw, train_loader_small, opt_sw, crit_sw, device)
        val_sw, _ = evaluate(m_sw, val_loader, crit_sw, device)
    do_sweep_rows.append({
        "p": p_sw, "Train Acc": tr_sw, "Val Acc": val_sw, "Gap": tr_sw - val_sw,
    })
    print(f"  p={p_sw:.1f}  Train={tr_sw:.4f}  Val={val_sw:.4f}  "
          f"Gap={tr_sw - val_sw:.4f}")

fig_ds, ax_ds = plt.subplots(figsize=(8, 4))
p_x = [r["p"] for r in do_sweep_rows]
ax_ds.plot(p_x, [r["Train Acc"] for r in do_sweep_rows],
           "o-", color="#1f77b4", label="Train Acc")
ax_ds.plot(p_x, [r["Val Acc"]   for r in do_sweep_rows],
           "s--", color="#d62728", label="Val Acc")
ax_ds.fill_between(p_x,
                   [r["Train Acc"] for r in do_sweep_rows],
                   [r["Val Acc"]   for r in do_sweep_rows],
                   alpha=0.15, color="orange", label="Gap")
ax_ds.set_xlabel("Dropout probability p", fontsize=11)
ax_ds.set_ylabel("Accuracy", fontsize=11)
ax_ds.set_title("Dropout Rate Sweep: Accuracy vs p (5K train subset)",
                fontsize=11, fontweight="bold")
ax_ds.legend(fontsize=9)
ax_ds.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_p = max(do_sweep_rows, key=lambda r: r["Val Acc"])
print(f"\nBest dropout p: {best_p['p']:.1f}  "
      f"Val Acc={best_p['Val Acc']:.4f}  Gap={best_p['Gap']:.4f}")


---
## Part 3 — Training on FashionMNIST

We train a deliberately overparameterised MLP (width=512, depth=3) on only 5,000 training samples — a recipe for strong overfitting. We compare six configurations:

| Config | Regularisation |
|--------|---------------|
| No-Reg | None |
| L1     | L1 penalty (λ=1e-4) on all weights |
| L2     | L2 weight decay (λ=1e-4) |
| Dropout | Dropout (p=0.4) after each hidden layer |
| BatchNorm | BatchNorm1d after each hidden linear |
| Combined | Dropout + L2 (best-practice combination) |

In [ ]:
# ── Regularised MLP ───────────────────────────────────────────────────────────

class FlattenView(nn.Module):
    '''Flatten spatial dims inside nn.Sequential.'''

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Flatten all dims except batch dim.

        Args:
            x: Input of shape (N, ...).

        Returns:
            Tensor of shape (N, prod(rest)).
        '''
        return x.view(x.size(0), -1)


class RegMLP(nn.Module):
    '''MLP with configurable regularisation: Dropout, BatchNorm, or neither.

    Attributes:
        net: Sequential layer stack.
        use_l1: Whether to add L1 penalty manually during training.
        lambda_l1: L1 regularisation strength.
    '''

    def __init__(
        self,
        input_dim:  int   = INPUT_DIM,
        hidden_dim: int   = HIDDEN_DIM,
        output_dim: int   = NUM_CLASSES,
        n_hidden:   int   = N_HIDDEN,
        dropout_p:  float = 0.0,
        batchnorm:  bool  = False,
        lambda_l1:  float = 0.0,
    ) -> None:
        '''Initialise RegMLP.

        Args:
            input_dim:  Number of input features.
            hidden_dim: Width of each hidden layer.
            output_dim: Number of output logits.
            n_hidden:   Number of hidden layers.
            dropout_p:  Dropout probability (0 = no dropout).
            batchnorm:  If True, insert BatchNorm1d after each linear.
            lambda_l1:  If > 0, L1 penalty is added to the loss during training.
        '''
        super().__init__()
        self.lambda_l1 = lambda_l1
        layers: list[nn.Module] = [FlattenView()]
        in_dim = input_dim
        for _ in range(n_hidden):
            layers.append(nn.Linear(in_dim, hidden_dim))
            if batchnorm:
                layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            if dropout_p > 0.0:
                layers.append(nn.Dropout(p=dropout_p))
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Compute logits.

        Args:
            x: Input of shape (N, input_dim) or (N, C, H, W).

        Returns:
            Logits of shape (N, output_dim).
        '''
        return self.net(x)

    def l1_penalty(self) -> torch.Tensor:
        '''Compute L1 penalty sum over all linear layer weights.

        Returns:
            Scalar tensor: lambda_l1 * sum(|w|) over all weight matrices.
        '''
        penalty = torch.tensor(0.0, device=next(self.parameters()).device)
        for module in self.net:
            if isinstance(module, nn.Linear):
                penalty = penalty + module.weight.abs().sum()
        return self.lambda_l1 * penalty


# ── Config table ──────────────────────────────────────────────────────────────
reg_configs: dict[str, dict] = {
    "No-Reg":   {"dropout_p": 0.0, "batchnorm": False, "lambda_l1": 0.0,      "weight_decay": 0.0},
    "L1":       {"dropout_p": 0.0, "batchnorm": False, "lambda_l1": LAMBDA_L1, "weight_decay": 0.0},
    "L2":       {"dropout_p": 0.0, "batchnorm": False, "lambda_l1": 0.0,       "weight_decay": LAMBDA_L2},
    "Dropout":  {"dropout_p": DROPOUT_P, "batchnorm": False, "lambda_l1": 0.0,  "weight_decay": 0.0},
    "BatchNorm":{"dropout_p": 0.0, "batchnorm": True,  "lambda_l1": 0.0,       "weight_decay": 0.0},
    "Combined": {"dropout_p": DROPOUT_P, "batchnorm": False, "lambda_l1": 0.0,  "weight_decay": LAMBDA_L2},
}
print("Regularisation configurations:")
for name, cfg in reg_configs.items():
    print(f"  {name:<12s}: dropout={cfg['dropout_p']}  BN={cfg['batchnorm']}"
          f"  L1={cfg['lambda_l1']}  L2={cfg['weight_decay']}")

In [ ]:
# ── Standard training functions ────────────────────────────────────────────────

def train_one_epoch(
    model:      nn.Module,
    dataloader: DataLoader,
    optimizer:  torch.optim.Optimizer,
    criterion:  nn.Module,
    device:     torch.device,
) -> tuple[float, float]:
    '''Train for one epoch, adding L1 penalty if model.lambda_l1 > 0.

    Args:
        model:      Neural network (RegMLP or compatible).
        dataloader: Training DataLoader.
        optimizer:  Optimiser instance.
        criterion:  Base loss function (CrossEntropyLoss).
        device:     Compute device.

    Returns:
        Tuple of (avg_loss, accuracy) for the epoch.
    '''
    model.train()
    running_loss = 0.0
    correct = 0
    total   = 0
    for batch_inputs, batch_targets in dataloader:
        batch_inputs  = batch_inputs.to(device)
        batch_targets = batch_targets.to(device)
        optimizer.zero_grad()
        outputs  = model(batch_inputs)
        loss     = criterion(outputs, batch_targets)
        # Add L1 penalty if configured
        if hasattr(model, "lambda_l1") and model.lambda_l1 > 0.0:
            loss = loss + model.l1_penalty()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_inputs.size(0)
        _, predicted  = outputs.max(1)
        total        += batch_targets.size(0)
        correct      += predicted.eq(batch_targets).sum().item()
    return running_loss / total, correct / total


def evaluate(
    model:      nn.Module,
    dataloader: DataLoader,
    criterion:  nn.Module,
    device:     torch.device,
) -> tuple[float, float]:
    '''Evaluate model.

    Args:
        model:      Neural network.
        dataloader: Validation or test DataLoader.
        criterion:  Loss function (CE only, no L1 added).
        device:     Compute device.

    Returns:
        Tuple of (avg_loss, accuracy).
    '''
    model.eval()
    running_loss = 0.0
    correct = 0
    total   = 0
    with torch.no_grad():
        for batch_inputs, batch_targets in dataloader:
            batch_inputs  = batch_inputs.to(device)
            batch_targets = batch_targets.to(device)
            outputs       = model(batch_inputs)
            loss          = criterion(outputs, batch_targets)
            running_loss += loss.item() * batch_inputs.size(0)
            _, predicted  = outputs.max(1)
            total        += batch_targets.size(0)
            correct      += predicted.eq(batch_targets).sum().item()
    return running_loss / total, correct / total

In [ ]:
# ── Train all six configurations ──────────────────────────────────────────────
criterion     = nn.CrossEntropyLoss()
reg_results:   list[dict]      = []
reg_histories: dict[str, dict] = {}

for config_name, cfg in reg_configs.items():
    model_r = RegMLP(
        dropout_p  = cfg["dropout_p"],
        batchnorm  = cfg["batchnorm"],
        lambda_l1  = cfg["lambda_l1"],
    ).to(device)
    opt_r = torch.optim.Adam(
        model_r.parameters(),
        lr           = LEARNING_RATE,
        weight_decay = cfg["weight_decay"],   # L2 via Adam weight_decay
    )
    history_r: dict[str, list[float]] = {
        "train_loss": [], "val_loss": [],
        "train_acc":  [], "val_acc":  [],
    }
    best_val_loss = float("inf")
    best_state    = None
    t_start_r     = time.time()

    for epoch in range(NUM_EPOCHS):
        tr_loss, tr_acc = train_one_epoch(
            model_r, train_loader_small, opt_r, criterion, device
        )
        v_loss, v_acc = evaluate(model_r, val_loader, criterion, device)
        history_r["train_loss"].append(tr_loss)
        history_r["val_loss"].append(v_loss)
        history_r["train_acc"].append(tr_acc)
        history_r["val_acc"].append(v_acc)
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_state    = {k: v.clone() for k, v in model_r.state_dict().items()}
        if (epoch + 1) % 10 == 0 or epoch == 0:
            elapsed_r = time.time() - t_start_r
            print(f"[{config_name:<10s}] Epoch {epoch+1:>2d}/{NUM_EPOCHS} | "
                  f"Train Acc: {tr_acc:.2%} | Val Acc: {v_acc:.2%} | "
                  f"Time: {elapsed_r:.1f}s")

    model_r.load_state_dict(best_state)
    _, test_acc_r = evaluate(model_r, test_loader, criterion, device)
    overfitting   = history_r["train_acc"][-1] - history_r["val_acc"][-1]
    reg_results.append({
        "Config":        config_name,
        "Train Acc":     round(history_r["train_acc"][-1], 4),
        "Val Acc":       round(history_r["val_acc"][-1],   4),
        "Test Acc":      round(test_acc_r, 4),
        "Overfit Gap":   round(overfitting, 4),
    })
    reg_histories[config_name] = history_r
    print()

In [ ]:
# ── Training curves: train vs val accuracy ────────────────────────────────────
colors_cfg = {
    "No-Reg":    "#d62728",
    "L1":        "#ff7f0e",
    "L2":        "#2ca02c",
    "Dropout":   "#1f77b4",
    "BatchNorm": "#9467bd",
    "Combined":  "#8c564b",
}
fig_tc, axes_tc = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, NUM_EPOCHS + 1)

for cfg_name, color in colors_cfg.items():
    hist = reg_histories[cfg_name]
    axes_tc[0].plot(epochs_range, hist["train_acc"], color=color, lw=2,
                    ls="--", alpha=0.7)
    axes_tc[0].plot(epochs_range, hist["val_acc"],   color=color, lw=2,
                    ls="-",  label=cfg_name)
    axes_tc[1].plot(epochs_range, hist["val_loss"],  color=color, lw=2,
                    marker="o", markersize=3, label=cfg_name)

axes_tc[0].set_xlabel("Epoch"); axes_tc[0].set_ylabel("Accuracy")
axes_tc[0].set_title("Accuracy — Train (dashed) vs Val (solid)",
                     fontsize=11, fontweight="bold")
axes_tc[0].legend(fontsize=8, ncol=2)
axes_tc[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes_tc[1].set_xlabel("Epoch"); axes_tc[1].set_ylabel("Validation Loss")
axes_tc[1].set_title("Validation Loss", fontsize=11, fontweight="bold")
axes_tc[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

results_df = pd.DataFrame(reg_results).sort_values("Val Acc", ascending=False)
print("Results (sorted by Val Acc):")
print(results_df.to_string(index=False))

---
## Part 4 — Evaluation & Analysis

### 4.1 Weight Distribution Analysis

We compare the distribution of learned weight magnitudes across regularisation strategies. L1 should produce sparse (spiky near zero) distributions; L2 should produce tight Gaussians; Dropout and BatchNorm should produce wider distributions.

In [ ]:
# ── Retrain all configs saving model weights ──────────────────────────────────
# (We reuse reg_histories and retrain to get final model weights)
trained_models: dict[str, nn.Module] = {}

for config_name, cfg in reg_configs.items():
    torch.manual_seed(SEED)
    model_w = RegMLP(
        dropout_p  = cfg["dropout_p"],
        batchnorm  = cfg["batchnorm"],
        lambda_l1  = cfg["lambda_l1"],
    ).to(device)
    opt_w = torch.optim.Adam(model_w.parameters(), lr=LEARNING_RATE,
                              weight_decay=cfg["weight_decay"])
    for _ in range(NUM_EPOCHS):
        train_one_epoch(model_w, train_loader_small, opt_w, criterion, device)
    trained_models[config_name] = model_w

# ── Extract and plot weight distributions ─────────────────────────────────────
fig_wd, axes_wd = plt.subplots(2, 3, figsize=(14, 8))
axes_flat_wd    = axes_wd.ravel()

for idx, (config_name, model_w) in enumerate(trained_models.items()):
    all_weights = np.concatenate([
        p.detach().cpu().numpy().ravel()
        for name, p in model_w.named_parameters()
        if "weight" in name and p.requires_grad
    ])
    ax_wd = axes_flat_wd[idx]
    ax_wd.hist(all_weights, bins=80, density=True,
               color=colors_cfg[config_name], alpha=0.7)
    n_near_zero = int((np.abs(all_weights) < 1e-3).sum())
    ax_wd.set_title(
        f"{config_name}\n"
        f"mean={all_weights.mean():.4f}  std={all_weights.std():.4f}"
        f"  |w|<1e-3: {n_near_zero:,}",
        fontsize=9, fontweight="bold"
    )
    ax_wd.set_xlabel("Weight value", fontsize=8)
    ax_wd.set_ylabel("Density", fontsize=8)
    ax_wd.tick_params(labelsize=7)

fig_wd.suptitle("Weight Distributions After Training — One Plot per Config",
                fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print("Weight statistics after training:")
print(f"{'Config':<12s}  {'Mean':>8s}  {'Std':>8s}  {'|w|_max':>9s}  {'Near-zero':>10s}")
print("-" * 54)
for config_name, model_w in trained_models.items():
    all_w = np.concatenate([
        p.detach().cpu().numpy().ravel()
        for nm, p in model_w.named_parameters()
        if "weight" in nm and p.requires_grad
    ])
    near_z = int((np.abs(all_w) < 1e-3).sum())
    print(f"{config_name:<12s}  {all_w.mean():>8.5f}  {all_w.std():>8.5f}"
          f"  {np.abs(all_w).max():>9.5f}  {near_z:>10,}")

### 4.2 Early Stopping

Early stopping halts training when the validation loss has not improved for `patience` consecutive epochs — it is the simplest and most universally effective regularisation technique, requiring no hyperparameter tuning beyond the patience value.

In [ ]:
# ── Early stopping on the no-regularisation model ────────────────────────────

class EarlyStopping:
    '''Monitor validation loss and signal when training should stop.

    Attributes:
        patience:   Number of epochs to wait before stopping.
        delta:      Minimum improvement to count as an improvement.
        best_loss:  Best validation loss seen so far.
        counter:    Number of epochs without improvement.
        best_state: State dict of the best model seen.
    '''

    def __init__(self, patience: int = PATIENCE, delta: float = 0.0) -> None:
        '''Initialise EarlyStopping.

        Args:
            patience: Epochs to wait without improvement before stopping.
            delta:    Minimum change to qualify as an improvement.
        '''
        self.patience   = patience
        self.delta      = delta
        self.best_loss  = float("inf")
        self.counter    = 0
        self.best_state: dict | None = None

    def step(self, val_loss: float, model: nn.Module) -> bool:
        '''Update stopping criteria and save best model state.

        Args:
            val_loss: Current epoch validation loss.
            model:    Model to checkpoint if this is the best epoch.

        Returns:
            True if training should stop, False otherwise.
        '''
        if val_loss < self.best_loss - self.delta:
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience


# ── Train for 40 epochs with early stopping (no regularisation) ───────────────
NUM_EPOCHS_ES = 40
torch.manual_seed(SEED)
model_es = RegMLP(dropout_p=0.0, batchnorm=False, lambda_l1=0.0).to(device)
opt_es   = torch.optim.Adam(model_es.parameters(), lr=LEARNING_RATE)
es       = EarlyStopping(patience=PATIENCE)

es_train_losses: list[float] = []
es_val_losses:   list[float] = []
stopped_epoch    = NUM_EPOCHS_ES

for epoch in range(NUM_EPOCHS_ES):
    tr_loss_es, _ = train_one_epoch(model_es, train_loader_small, opt_es, criterion, device)
    v_loss_es, _  = evaluate(model_es, val_loader, criterion, device)
    es_train_losses.append(tr_loss_es)
    es_val_losses.append(v_loss_es)
    if es.step(v_loss_es, model_es):
        stopped_epoch = epoch + 1
        print(f"Early stopping triggered at epoch {stopped_epoch}/{NUM_EPOCHS_ES}.")
        break

# Restore best model
if es.best_state is not None:
    model_es.load_state_dict(es.best_state)

_, es_val_acc  = evaluate(model_es, val_loader,  criterion, device)
_, es_test_acc = evaluate(model_es, test_loader, criterion, device)
print(f"Best val loss: {es.best_loss:.4f}  "
      f"Val Acc: {es_val_acc:.2%}  Test Acc: {es_test_acc:.2%}")

# ── Plot early stopping ────────────────────────────────────────────────────────
fig_es, ax_es = plt.subplots(figsize=(9, 4))
epochs_es = range(1, len(es_train_losses) + 1)
ax_es.plot(epochs_es, es_train_losses, color="#ff7f0e", lw=2, label="Train Loss")
ax_es.plot(epochs_es, es_val_losses,   color="#1f77b4", lw=2, label="Val Loss")
ax_es.axvline(stopped_epoch, color="red", lw=1.5, ls="--",
              label=f"Stopped at epoch {stopped_epoch}")
ax_es.axvline(es.best_loss, color="green", lw=0, ls="-")  # invisible — just for legend
best_ep = np.argmin(es_val_losses) + 1
ax_es.axvline(best_ep, color="#2ca02c", lw=1.5, ls=":",
              label=f"Best val loss at epoch {best_ep}")
ax_es.set_xlabel("Epoch"); ax_es.set_ylabel("Loss")
ax_es.set_title("Early Stopping on No-Regularisation Model (40 epochs)",
                fontsize=11, fontweight="bold")
ax_es.legend(fontsize=9)
plt.tight_layout()
plt.show()
print(f"Training diverged after epoch {best_ep} (val loss began rising).")
print(f"Early stopping saved the model from {NUM_EPOCHS_ES - stopped_epoch} extra epochs.")

In [ ]:
# ── L2 regularisation strength sweep ─────────────────────────────────────────
l2_strengths   = [0.0, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2]
ablation_rows: list[dict] = []

for l2_val in l2_strengths:
    torch.manual_seed(SEED)
    model_ab = RegMLP(dropout_p=0.0, batchnorm=False, lambda_l1=0.0).to(device)
    opt_ab   = torch.optim.Adam(model_ab.parameters(), lr=LEARNING_RATE,
                                weight_decay=l2_val)
    best_val_ab = float("inf")
    best_st_ab  = None
    for _ in range(NUM_EPOCHS):
        train_one_epoch(model_ab, train_loader_small, opt_ab, criterion, device)
        v_loss_ab, v_acc_ab = evaluate(model_ab, val_loader, criterion, device)
        if v_loss_ab < best_val_ab:
            best_val_ab = v_loss_ab
            best_st_ab  = {k: v.clone() for k, v in model_ab.state_dict().items()}
    model_ab.load_state_dict(best_st_ab)
    tr_loss_ab, tr_acc_ab = evaluate(model_ab, train_loader_small, criterion, device)
    _, te_acc_ab           = evaluate(model_ab, test_loader, criterion, device)
    ablation_rows.append({
        "L2 lambda":   l2_val,
        "Train Acc":   round(tr_acc_ab, 4),
        "Val Acc":     round(v_acc_ab, 4),
        "Test Acc":    round(te_acc_ab, 4),
        "Overfit Gap": round(tr_acc_ab - v_acc_ab, 4),
    })
    print(f"L2={l2_val:<8}  Train={tr_acc_ab:.2%}  Val={v_acc_ab:.2%}  "
          f"Gap={tr_acc_ab - v_acc_ab:.2%}")

abl_df = pd.DataFrame(ablation_rows)
print("\nL2 ablation summary:")
print(abl_df.to_string(index=False))

# ── Plot ablation ──────────────────────────────────────────────────────────────
fig_ab, ax_ab = plt.subplots(figsize=(9, 4))
ax_ab.semilogx([r["L2 lambda"] if r["L2 lambda"] > 0 else 1e-6 for r in ablation_rows],
               [r["Val Acc"]  for r in ablation_rows],
               color="#1f77b4", lw=2, marker="o", markersize=7, label="Val Acc")
ax_ab.semilogx([r["L2 lambda"] if r["L2 lambda"] > 0 else 1e-6 for r in ablation_rows],
               [r["Train Acc"] for r in ablation_rows],
               color="#ff7f0e", lw=2, marker="s", markersize=7, ls="--", label="Train Acc")
ax_ab.set_xlabel("L2 regularisation strength λ (log scale)", fontsize=11)
ax_ab.set_ylabel("Accuracy", fontsize=11)
ax_ab.set_title("L2 Regularisation Strength Ablation", fontsize=11, fontweight="bold")
ax_ab.legend(fontsize=9)
ax_ab.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax_ab.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()
print("Optimal λ is where val acc peaks. Too small → overfit. Too large → underfit.")

### 4.3 L2 Regularisation Strength Ablation

We sweep the L2 weight-decay coefficient over eight values spanning six orders of magnitude. This makes the bias–variance trade-off concrete: at $\lambda=0$ the model overfits; at large $\lambda$ it underfits; somewhere in between lies the optimal regularisation strength.


In [ ]:
# ── L2 Strength Ablation: bias-variance trade-off across lambda values ────────
# Sweep lambda in {0, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2}.
# Use weight_decay in Adam optimizer (equivalent to L2 penalty in SGD).

lambda_values: list[float] = [0.0, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2]
l2_abl_rows:   list[dict]  = []

for lam_val in lambda_values:
    abl_model = RegMLP(
        input_dim   = 784,
        hidden_dim  = HIDDEN_DIM,
        n_hidden    = N_HIDDEN,
        num_classes = NUM_CLASSES,
        dropout_p   = 0.0,
        use_batchnorm = False,
        lambda_l1   = 0.0,
    ).to(device)
    opt_abl  = torch.optim.Adam(
        abl_model.parameters(), lr=LEARNING_RATE, weight_decay=lam_val
    )
    crit_abl = nn.CrossEntropyLoss()
    tr_acc_abl, val_acc_abl = 0.0, 0.0
    for _ep in range(NUM_EPOCHS):
        tr_acc_abl           = train_one_epoch(
            abl_model, train_loader_small, opt_abl, crit_abl, device
        )
        val_acc_abl, _       = evaluate(abl_model, val_loader, crit_abl, device)
    l2_abl_rows.append({
        "lambda":    lam_val,
        "Train Acc": tr_acc_abl,
        "Val Acc":   val_acc_abl,
        "Gap":       tr_acc_abl - val_acc_abl,
    })
    print(f"lambda={lam_val:>8.1e}  Train={tr_acc_abl:.4f}  Val={val_acc_abl:.4f}"
          f"  Gap={tr_acc_abl - val_acc_abl:.4f}")

# ── Plot: val accuracy and train-val gap vs lambda ────────────────────────────
fig_l2, axes_l2 = plt.subplots(1, 2, figsize=(13, 4))

lam_lbls = [f"{r['lambda']:.1e}" for r in l2_abl_rows]
x_l2     = list(range(len(l2_abl_rows)))

axes_l2[0].plot(x_l2, [r["Train Acc"] for r in l2_abl_rows],
                "o-", color="#1f77b4", label="Train Acc")
axes_l2[0].plot(x_l2, [r["Val Acc"]   for r in l2_abl_rows],
                "s--", color="#d62728", label="Val Acc")
axes_l2[0].set_xticks(x_l2)
axes_l2[0].set_xticklabels(lam_lbls, rotation=30, fontsize=9)
axes_l2[0].set_xlabel("L2 weight_decay", fontsize=11)
axes_l2[0].set_ylabel("Accuracy", fontsize=11)
axes_l2[0].set_title("L2 Ablation: Train vs Val Accuracy",
                      fontsize=11, fontweight="bold")
axes_l2[0].legend(fontsize=9)
axes_l2[0].grid(alpha=0.3)

gap_vals_l2 = [r["Gap"] for r in l2_abl_rows]
bar_cols_l2 = [
    "#2ca02c" if g < 0.05 else "#ff7f0e" if g < 0.10 else "#d62728"
    for g in gap_vals_l2
]
axes_l2[1].bar(x_l2, gap_vals_l2, color=bar_cols_l2, edgecolor="k", lw=0.7)
axes_l2[1].axhline(0.05, color="green", lw=1, ls="--", label="5% gap threshold")
axes_l2[1].set_xticks(x_l2)
axes_l2[1].set_xticklabels(lam_lbls, rotation=30, fontsize=9)
axes_l2[1].set_xlabel("L2 weight_decay", fontsize=11)
axes_l2[1].set_ylabel("Train - Val Acc", fontsize=11)
axes_l2[1].set_title("L2 Ablation: Generalisation Gap",
                      fontsize=11, fontweight="bold")
axes_l2[1].legend(fontsize=9)
axes_l2[1].grid(alpha=0.3)

plt.suptitle("Effect of L2 Regularisation Strength on Bias-Variance Trade-off",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Best lambda summary ───────────────────────────────────────────────────────
best_l2 = max(l2_abl_rows, key=lambda r: r["Val Acc"])
print(f"\nBest lambda: {best_l2['lambda']:.1e}  "
      f"Val Acc={best_l2['Val Acc']:.4f}  Gap={best_l2['Gap']:.4f}")
print("\nPattern:")
print("  lambda=0       -> highest train acc, largest gap  (overfitting)")
print("  small lambda   -> minimal gap with little accuracy loss  (sweet spot)")
print("  large lambda   -> lower train AND val accuracy  (underfitting)")


---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Overfitting is the enemy**: a wide, deep MLP trained on 5,000 samples achieves near-100%    training accuracy but poor validation accuracy. Every technique here reduces this gap.
2. **L1 vs L2 differ in sparsity**: L1 drives small weights to exactly zero (useful for feature    selection); L2 keeps all weights small but non-zero (better for dense feature sets like image pixels).
3. **Dropout forces redundant representations**: by randomly silencing neurons, the network cannot    rely on any single neuron being active — each neuron must independently encode useful information.
4. **Batch Normalisation reduces internal covariate shift**: normalising activations per mini-batch    stabilises the training signal, allows higher learning rates, and acts as an implicit regulariser    by introducing noise proportional to the batch variance.
5. **Early stopping is the most universally applicable technique**: it requires no architectural    changes, no additional hyperparameters (beyond `patience`), and consistently prevents overfitting    at no cost to validation accuracy.

### What's Next

→ **05-06 (Backpropagation)** derives the chain rule and implements manual gradients through the MLP layers we have been training — revealing the machinery that `loss.backward()` executes.

### Going Further

- Srivastava et al. (2014) — *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*.
- Ioffe & Szegedy (2015) — *Batch Normalization: Accelerating Deep Network Training*.
- Goodfellow, Bengio & Courville (2016) — *Deep Learning*, Chapter 7 (Regularisation).